# Phase 11 Submission Readiness

This notebook executes the authorized Phase 11 Option C scope only: final refit on full train, test inference, and validated candidate submissions for CatBoost tuned and M1 Logistic Regression baseline.

It does not upload, use leaderboard feedback, declare a final winner, declare a submission-ready model before independent review, run HPO, create ensembles, tune thresholds, fit calibration, use School as a feature, use external data, or modify `logs/experiment_log.csv`.


## 1. Setup And Authorization Gate

**Objective.** Verify the authorized commit, load the authorization note, and define run paths.
**Inputs.** Git metadata and Phase 11 authorization/planning documents.
**Method.** Fail closed on missing authorization, wrong HEAD, or output path collision.
**Expected output.** Stable paths, run identifiers, and authorization metadata.
**Risk controlled.** Prevents running from an unauthorized state or overwriting prior artifacts.


In [1]:
# 1.1 Imports, constants, and path setup
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_SEED = 42
EXPERIMENT_ID = "phase11_submission_readiness_v1"
RUN_ID = "phase11_option_c_20260619_0001"
AUTHORIZED_HEAD = "e5ea4e8029ad0a1eb8cbe7260a0a949b5f4beb48"
PLANNING_BASELINE_HEAD = "6ffb66d7165361a7c3247757edad7da134eba2ad"
PHASE10_ACCEPTANCE_HEAD = "de11fae08a511a5a230a12cecb29d980aa60af74"
REPO_ROOT = Path(r"C:\GitHub\reto_Tokio")
assert (REPO_ROOT / ".git").exists(), "REPO_ROOT must point to the repository root"

DATA_DIR = REPO_ROOT / "data" / "input"
OUTPUTS_DIR = REPO_ROOT / "outputs"
WAVE2_PY = Path(r"C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe")

paths = {
    "notebook": REPO_ROOT / "notebooks" / "11_phase11_submission_readiness.ipynb",
    "candidate_selection": OUTPUTS_DIR / "validation" / f"phase11_submission_readiness_{RUN_ID}_candidate_selection_report.csv",
    "final_refit": OUTPUTS_DIR / "validation" / f"phase11_submission_readiness_{RUN_ID}_final_refit_report.csv",
    "submission_validation": OUTPUTS_DIR / "validation" / f"phase11_submission_readiness_{RUN_ID}_submission_validation.csv",
    "model_summary": OUTPUTS_DIR / "validation" / f"phase11_submission_readiness_{RUN_ID}_model_summary.csv",
    "validation_report": OUTPUTS_DIR / "reports" / f"phase11_submission_readiness_{RUN_ID}_validation_report.md",
    "artifact_manifest": OUTPUTS_DIR / "reports" / f"phase11_submission_readiness_{RUN_ID}_artifact_manifest.csv",
    "candidate_log": OUTPUTS_DIR / "reports" / f"phase11_submission_readiness_{RUN_ID}_experiment_log_candidate.csv",
    "catboost_submission": OUTPUTS_DIR / "submissions" / f"phase11_submission_readiness_{RUN_ID}_catboost_tuned_submission.csv",
    "m1_submission": OUTPUTS_DIR / "submissions" / f"phase11_submission_readiness_{RUN_ID}_m1_logistic_regression_baseline_submission.csv",
}

AUTH_NOTE = REPO_ROOT / "docs" / "11_submission_readiness" / "phase11_project_authorization_note.md"
REQUIRED_DOCS = [
    AUTH_NOTE,
    REPO_ROOT / "docs" / "11_submission_readiness" / "phase11_master_planning_brief.md",
    REPO_ROOT / "docs" / "11_submission_readiness" / "phase11_operator_runbook.md",
    REPO_ROOT / "docs" / "11_submission_readiness" / "prompt_codex_phase11_submission_readiness_execution.md",
    REPO_ROOT / "docs" / "11_submission_readiness" / "prompt_opus_phase11_final_submission_review.md",
    REPO_ROOT / "docs" / "10_model_optimization" / "phase10_acceptance.md",
    REPO_ROOT / "docs" / "05_methodology" / "validation_protocol_phase6.md",
    REPO_ROOT / "docs" / "05_methodology" / "leakage_checklist_phase6.md",
    REPO_ROOT / "docs" / "00_project_contract" / "challenge_brief.md",
    REPO_ROOT / "docs" / "00_project_contract" / "submission_checklist.md",
]
missing_docs = [str(p.relative_to(REPO_ROOT)) for p in REQUIRED_DOCS if not p.exists()]
if missing_docs:
    raise FileNotFoundError(f"Missing required predecessor documents: {missing_docs}")
auth_text = AUTH_NOTE.read_text(encoding="utf-8")
for required_phrase in [
    "Option C",
    "Written waiver granted",
    "CatBoost tuned",
    "M1 Logistic Regression baseline",
    "No agent uploads",
]:
    if required_phrase not in auth_text:
        raise RuntimeError(f"Authorization note missing required phrase: {required_phrase}")

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
short_head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip()
if head != AUTHORIZED_HEAD:
    raise RuntimeError(f"Unauthorized HEAD: {head}; expected {AUTHORIZED_HEAD}")

staged = subprocess.check_output(["git", "diff", "--cached", "--name-only"], cwd=REPO_ROOT, text=True).strip()
if staged:
    raise RuntimeError(f"Unexpected staged files before execution: {staged}")

log_before = (REPO_ROOT / "logs" / "experiment_log.csv").read_bytes()
for key, path in paths.items():
    if key == "notebook":
        continue
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)

print({"head": head, "run_id": RUN_ID, "authorization": "verified", "output_paths_checked": len(paths) - 1})


{'head': 'e5ea4e8029ad0a1eb8cbe7260a0a949b5f4beb48', 'run_id': 'phase11_option_c_20260619_0001', 'authorization': 'verified', 'output_paths_checked': 9}


### Interpretation

- **Main result:** Authorization, HEAD and output collision gates passed.
- **Methodological reading:** Execution is bound to the committed authorization note and Option C.
- **Risk or warning:** The authorization note references `6ffb66d` as planning baseline; operational HEAD is the documented successor `e5ea4e8`.
- **Diagnostic decision:** Continue to data and feature contract checks.


## 2. Official Data And F2 Feature Contract

**Objective.** Load official data only and construct F2 identically for full train and test.
**Inputs.** `train.csv`, `test.csv`, `sample_submission.csv`.
**Method.** Build row-wise missingness flags and `available_measurement_count`; assert School exclusion.
**Expected output.** Full-train and test feature matrices with the authorized 21 F2 features.
**Risk controlled.** Prevents School leakage, external data usage, and train/test feature drift.


In [2]:
# 2.1 Load official files and build F2 features
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

if train.shape != (2781, 16):
    raise RuntimeError(f"Unexpected train shape: {train.shape}")
if test.shape != (696, 15):
    raise RuntimeError(f"Unexpected test shape: {test.shape}")
if sample.shape != (696, 2):
    raise RuntimeError(f"Unexpected sample_submission shape: {sample.shape}")
if "Drafted" not in train.columns:
    raise RuntimeError("train.csv must contain Drafted target")
if "Drafted" in test.columns:
    raise RuntimeError("test.csv must not contain Drafted target")
if list(sample.columns) != ["Id", "Drafted"]:
    raise RuntimeError("sample_submission.csv must have columns Id,Drafted")
if not test["Id"].equals(sample["Id"]):
    raise RuntimeError("test.csv Id order must match sample_submission.csv Id order")

base_numeric = [
    "Year",
    "Age",
    "Height",
    "Weight",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
role_categorical = ["Player_Type", "Position_Type", "Position"]
physical_test_columns = [
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle",
]
missing_flags = ["Age_missing"] + [f"{col}_missing" for col in physical_test_columns]
numeric_features = base_numeric + missing_flags + ["available_measurement_count"]
categorical_features = role_categorical
feature_columns = numeric_features + categorical_features
if len(feature_columns) != 21:
    raise RuntimeError(f"F2 feature count mismatch: {len(feature_columns)}")
if "School" in feature_columns or "Drafted" in feature_columns or "Id" in feature_columns:
    raise RuntimeError("F2 feature matrix contains forbidden Id/Drafted/School")

required_columns = ["Id", "School", "Drafted"] + base_numeric + role_categorical
missing_train = [col for col in required_columns if col not in train.columns]
missing_test = [col for col in ["Id", "School"] + base_numeric + role_categorical if col not in test.columns]
if missing_train or missing_test:
    raise RuntimeError(f"Missing required columns: train={missing_train}, test={missing_test}")

def build_f2(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    for col in physical_test_columns:
        out[f"{col}_missing"] = out[col].isna().astype(int)
    out["available_measurement_count"] = out[physical_test_columns].notna().sum(axis=1).astype(int)
    f2 = out[feature_columns].copy()
    if "School" in f2.columns:
        raise RuntimeError("School leaked into F2 feature matrix")
    return f2

X_train = build_f2(train)
X_test = build_f2(test)
y = train["Drafted"].astype(int)
if sorted(y.unique().tolist()) != [0, 1]:
    raise RuntimeError("Target must be binary 0/1")
if list(X_train.columns) != list(X_test.columns):
    raise RuntimeError("Train/test feature columns differ")
if list(X_train.columns) != feature_columns:
    raise RuntimeError("F2 feature order mismatch")

print({"train_rows": len(train), "test_rows": len(test), "features": len(feature_columns), "school_in_features": "School" in feature_columns})


{'train_rows': 2781, 'test_rows': 696, 'features': 21, 'school_in_features': False}


### Interpretation

- **Main result:** Official train/test/sample files and the F2 feature matrix passed contract checks.
- **Methodological reading:** Missingness and measurement availability are row-wise only; no target, School, or test fitting is used.
- **Risk or warning:** Test labels are unavailable by design; Phase 11 validates format and lineage, not test AUC.
- **Diagnostic decision:** Continue to candidate-selection and final-refit setup.


## 3. Candidate Selection And Hyperparameter Recovery

**Objective.** Recover the authorized candidate roles and hyperparameters without reselecting models.
**Inputs.** Phase 10 accepted reports and Phase 8 M1 baseline candidate log.
**Method.** Read persisted evidence and assert the authorization choices.
**Expected output.** Candidate-selection report and fixed model configs.
**Risk controlled.** Avoids guessing M1 baseline config or promoting a model from test/leaderboard information.


In [3]:
# 3.1 Recover accepted candidate evidence and fixed configs
phase10_summary = pd.read_csv(REPO_ROOT / "outputs" / "validation" / "phase10_model_optimization_phase10_standard_20260619_0152_model_summary.csv")
phase8_candidate_log = pd.read_csv(REPO_ROOT / "outputs" / "reports" / "phase8_model_family_comparison_v1_experiment_log_candidate.csv")

def summary_value(model_key: str, col: str) -> float:
    return float(phase10_summary.loc[phase10_summary["model_key"] == model_key, col].iloc[0])

m0_auc = summary_value("m0_random_forest_frozen", "oof_auc")
m1_auc = summary_value("m1_logistic_regression_baseline", "oof_auc")
cat_auc = summary_value("catboost_tuned", "oof_auc")
m1_tuned_auc = summary_value("m1_logistic_regression_tuned", "oof_auc")

m1_row = phase8_candidate_log.loc[phase8_candidate_log["model_key"] == "m1_logistic_regression"].iloc[0]
m1_params = json.loads(m1_row["model_params_summary"])
if m1_params.get("C") != 1.0 or m1_params.get("solver") != "lbfgs" or int(m1_params.get("max_iter")) != 1000:
    raise RuntimeError(f"Unexpected M1 baseline params recovered from Phase 8: {m1_params}")

cat_params = {
    "depth": 6,
    "learning_rate": 0.01,
    "l2_leaf_reg": 9,
    "iterations": 800,
    "border_count": 128,
    "random_seed": PROJECT_SEED,
    "cat_features": [],
}

candidate_selection_df = pd.DataFrame([
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "catboost_tuned",
        "role": "primary_final_refit_candidate_warning_heavy",
        "authorized": True,
        "oof_auc": cat_auc,
        "delta_vs_m0": cat_auc - m0_auc,
        "delta_vs_m1_baseline": cat_auc - m1_auc,
        "stability_gate": "written_waiver_granted",
        "final_winner_declared": False,
        "submission_ready_declared": False,
    },
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "m1_logistic_regression_baseline",
        "role": "fallback_reference_candidate",
        "authorized": True,
        "oof_auc": m1_auc,
        "delta_vs_m0": m1_auc - m0_auc,
        "delta_vs_m1_baseline": 0.0,
        "stability_gate": "not_applicable",
        "final_winner_declared": False,
        "submission_ready_declared": False,
    },
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "m1_logistic_regression_tuned",
        "role": "rejected_noise_level",
        "authorized": False,
        "oof_auc": m1_tuned_auc,
        "delta_vs_m0": m1_tuned_auc - m0_auc,
        "delta_vs_m1_baseline": m1_tuned_auc - m1_auc,
        "stability_gate": "not_applicable",
        "final_winner_declared": False,
        "submission_ready_declared": False,
    },
])

print(candidate_selection_df[["candidate", "role", "authorized", "oof_auc"]].to_string(index=False))


                      candidate                                        role  authorized  oof_auc
                 catboost_tuned primary_final_refit_candidate_warning_heavy        True 0.830321
m1_logistic_regression_baseline                fallback_reference_candidate        True 0.827082
   m1_logistic_regression_tuned                        rejected_noise_level       False 0.827482


### Interpretation

- **Main result:** Option C was resolved exactly: CatBoost tuned plus M1 baseline.
- **Methodological reading:** Candidate selection uses accepted OOF evidence only; no test or leaderboard signal is involved.
- **Risk or warning:** CatBoost remains warning-heavy and is covered by a written waiver, not by a winner declaration.
- **Diagnostic decision:** Proceed to full-train refits for both authorized candidates.


## 4. Full-Train Final Refit And Test Inference

**Objective.** Fit the two authorized candidates on full train and infer test probabilities.
**Inputs.** F2 train/test matrices, fixed configs, and the separate Wave 2 CatBoost environment.
**Method.** Fit learned preprocessing on full train only; transform test only; verify positive class probability direction.
**Expected output.** Two validated probability vectors for the official test rows.
**Risk controlled.** Prevents test fitting, wrong probability column extraction, and unauthorized model usage.


In [4]:
# 4.1 Fit M1 baseline on full train and generate test probabilities
def positive_class_proba(estimator, X_mat) -> np.ndarray:
    classes = list(estimator.classes_)
    if classes.count(1) != 1:
        raise RuntimeError(f"Estimator classes_ must contain label 1 exactly once, got {classes}")
    class_idx = classes.index(1)
    proba = estimator.predict_proba(X_mat)[:, class_idx]
    if not np.isfinite(proba).all() or not ((proba >= 0).all() and (proba <= 1).all()):
        raise RuntimeError("Predicted probabilities must be finite and in [0, 1]")
    return proba

m1_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)
m1_model = LogisticRegression(max_iter=1000, random_state=PROJECT_SEED)
m1_pipeline = Pipeline([
    ("preprocess", m1_preprocessor),
    ("model", m1_model),
])
m1_pipeline.fit(X_train, y)
m1_test_proba = positive_class_proba(m1_pipeline.named_steps["model"], m1_pipeline.named_steps["preprocess"].transform(X_test))
if len(m1_test_proba) != len(test):
    raise RuntimeError("M1 test probability row count mismatch")

print({"m1_rows": len(m1_test_proba), "m1_min": float(m1_test_proba.min()), "m1_max": float(m1_test_proba.max())})


{'m1_rows': 696, 'm1_min': 0.0010409329561047334, 'm1_max': 0.9894261701960908}


In [5]:
# 4.2 Fit CatBoost tuned in the separate Wave 2 environment and write its submission
if not WAVE2_PY.exists():
    raise FileNotFoundError(f"Separate Wave 2 Python not found: {WAVE2_PY}")

catboost_code = r'''
import json
import os
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

repo = Path(r"C:\GitHub\reto_Tokio")
data_dir = repo / "data" / "input"
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample = pd.read_csv(data_dir / "sample_submission.csv")

base_numeric = ["Year","Age","Height","Weight","Sprint_40yd","Vertical_Jump","Bench_Press_Reps","Broad_Jump","Agility_3cone","Shuttle"]
role_categorical = ["Player_Type","Position_Type","Position"]
physical_test_columns = ["Sprint_40yd","Vertical_Jump","Bench_Press_Reps","Broad_Jump","Agility_3cone","Shuttle"]
missing_flags = ["Age_missing"] + [f"{col}_missing" for col in physical_test_columns]
numeric_features = base_numeric + missing_flags + ["available_measurement_count"]
categorical_features = role_categorical
feature_columns = numeric_features + categorical_features
if "School" in feature_columns:
    raise RuntimeError("School leaked into CatBoost feature matrix")

def build_f2(df):
    out = df.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    for col in physical_test_columns:
        out[f"{col}_missing"] = out[col].isna().astype(int)
    out["available_measurement_count"] = out[physical_test_columns].notna().sum(axis=1).astype(int)
    return out[feature_columns].copy()

X_train = build_f2(train)
X_test = build_f2(test)
y = train["Drafted"].astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)
X_train_enc = preprocessor.fit_transform(X_train, y)
X_test_enc = preprocessor.transform(X_test)

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    depth=6,
    learning_rate=0.01,
    l2_leaf_reg=9,
    iterations=800,
    border_count=128,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)
model.fit(X_train_enc, y, cat_features=[])
classes = [int(value) for value in list(model.classes_)]
if classes.count(1) != 1:
    raise RuntimeError(f"CatBoost classes_ must contain label 1 exactly once, got {classes}")
idx = classes.index(1)
proba = model.predict_proba(X_test_enc)[:, idx]
if not np.isfinite(proba).all() or not ((proba >= 0).all() and (proba <= 1).all()):
    raise RuntimeError("CatBoost probabilities must be finite and in [0, 1]")
if len(proba) != len(test):
    raise RuntimeError("CatBoost row count mismatch")

submission = pd.DataFrame({"Id": sample["Id"].astype(int), "Drafted": proba})
out_path = Path(r"''' + str(paths["catboost_submission"]) + r'''")
submission.to_csv(out_path, index=False)
print(json.dumps({
    "candidate": "catboost_tuned",
    "rows": int(len(submission)),
    "classes": classes,
    "min_proba": float(proba.min()),
    "max_proba": float(proba.max()),
    "catboost_version": __import__("catboost").__version__,
}))
'''
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
cat_proc = subprocess.run(
    [str(WAVE2_PY), "-c", catboost_code],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    env=env,
)
if cat_proc.returncode != 0:
    raise RuntimeError(f"CatBoost separate-env run failed:\nSTDOUT:\n{cat_proc.stdout}\nSTDERR:\n{cat_proc.stderr}")
catboost_meta = json.loads(cat_proc.stdout.strip().splitlines()[-1])
print(catboost_meta)


{'candidate': 'catboost_tuned', 'rows': 696, 'classes': [0, 1], 'min_proba': 0.009068920828312368, 'max_proba': 0.9668055592272972, 'catboost_version': '1.2.10'}


### Interpretation

- **Main result:** Both authorized candidates were refit on full train and generated test probabilities.
- **Methodological reading:** Full-train fit is allowed only after authorization; test rows were transformed/inferred only.
- **Risk or warning:** CatBoost still carries the waiver-defined slice warnings from Phase 10; this notebook does not measure test AUC.
- **Diagnostic decision:** Build and validate the two candidate submission artifacts.


## 5. Submission Build And Validation Suite

**Objective.** Create and validate the two authorized submission CSV artifacts.
**Inputs.** Test Ids, sample submission, and candidate probability vectors.
**Method.** Check schema, 696 rows, Id set/order, numeric range, NaN/inf, duplicates, and SHA-256.
**Expected output.** Two validated CSVs and a submission validation report.
**Risk controlled.** Prevents invalid submission format, wrong Id order, manual edits, and untraceable files.


In [6]:
# 5.1 Build submissions and run validation checks
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in df[columns].iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, separator] + rows)

def safe_write_csv(df: pd.DataFrame, path: Path) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing artifact: {path}")
    df.to_csv(path, index=False)

m1_submission = pd.DataFrame({"Id": sample["Id"].astype(int), "Drafted": m1_test_proba})
safe_write_csv(m1_submission, paths["m1_submission"])

def validate_submission(candidate: str, path: Path) -> dict:
    df = pd.read_csv(path)
    checks = {
        "columns_exact": list(df.columns) == ["Id", "Drafted"],
        "row_count_696": len(df) == 696,
        "row_count_matches_test": len(df) == len(test) == len(sample),
        "id_set_matches": set(df["Id"].astype(int)) == set(test["Id"].astype(int)) == set(sample["Id"].astype(int)),
        "id_order_matches_sample": df["Id"].astype(int).reset_index(drop=True).equals(sample["Id"].astype(int).reset_index(drop=True)),
        "drafted_numeric": pd.api.types.is_numeric_dtype(df["Drafted"]),
        "drafted_in_unit_interval": df["Drafted"].between(0, 1).all(),
        "no_nan": not df["Drafted"].isna().any(),
        "no_inf": np.isfinite(df["Drafted"].to_numpy(dtype=float)).all(),
        "no_duplicate_id": df["Id"].nunique() == 696,
    }
    status = "pass" if all(checks.values()) else "fail"
    if status != "pass":
        raise RuntimeError(f"Submission validation failed for {candidate}: {checks}")
    file_sha = sha256_file(path)
    return {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": candidate,
        "submission_path": str(path.relative_to(REPO_ROOT)),
        "submission_rows": len(df),
        "min_probability": float(df["Drafted"].min()),
        "max_probability": float(df["Drafted"].max()),
        "submission_sha256": file_sha,
        "manual_edit_detected": False,
        "uploaded": False,
        "validation_status": status,
        **checks,
    }

validation_records = [
    validate_submission("catboost_tuned", paths["catboost_submission"]),
    validate_submission("m1_logistic_regression_baseline", paths["m1_submission"]),
]
submission_validation_df = pd.DataFrame(validation_records)
print(submission_validation_df[["candidate", "submission_rows", "validation_status", "submission_sha256"]].to_string(index=False))


                      candidate  submission_rows validation_status                                                submission_sha256
                 catboost_tuned              696              pass a6f14ef1a79aa883c6455de05fc16132bc9264c2d4ed59407aa86db9ec194cc8
m1_logistic_regression_baseline              696              pass 0804613d63c00d353497d20bd8a341721684f645499e2012a99f120557200640


### Interpretation

- **Main result:** Both candidate submissions passed the full validation suite.
- **Methodological reading:** The files are technically valid candidate artifacts, not uploaded submissions and not winner declarations.
- **Risk or warning:** Upload order remains a manual project-director decision.
- **Diagnostic decision:** Write lineage reports, manifests, and candidate logs.


## 6. Reports, Manifest, Candidate Log And Final Safety Checks

**Objective.** Write all authorized Phase 11 artifacts and verify forbidden paths remain untouched.
**Inputs.** Candidate submissions, validation records, configs, and lineage metadata.
**Method.** Create reports, SHA-256 manifest, candidate log, then run Git/path safety checks.
**Expected output.** Versioned artifacts plus final execution summary.
**Risk controlled.** Preserves traceability and prevents accidental main-log, forbidden-path, or leaderboard/upload side effects.


In [7]:
# 6.1 Write reports, manifest, and candidate log
created_at = datetime.now(timezone.utc).isoformat()

final_refit_df = pd.DataFrame([
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "catboost_tuned",
        "model_family": "CatBoostClassifier",
        "fit_scope": "full_train_only",
        "train_rows": len(train),
        "test_rows": len(test),
        "features_used": "F2",
        "school_used_as_feature": False,
        "hyperparameters_source": "Phase 10 hpo_results accepted config",
        "hyperparameters": json.dumps(cat_params, sort_keys=True),
        "environment": f"separate_wave2_env_python={WAVE2_PY}; catboost={catboost_meta.get('catboost_version')}",
    },
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "m1_logistic_regression_baseline",
        "model_family": "LogisticRegression",
        "fit_scope": "full_train_only",
        "train_rows": len(train),
        "test_rows": len(test),
        "features_used": "F2",
        "school_used_as_feature": False,
        "hyperparameters_source": "Phase 8 model-family comparison accepted baseline",
        "hyperparameters": json.dumps({"C": 1.0, "solver": "lbfgs", "max_iter": 1000, "random_state": PROJECT_SEED, "class_weight": None, "scaling": "StandardScaler on numeric features"}, sort_keys=True),
        "environment": f"base_venv_python={sys.executable}; sklearn={__import__('sklearn').__version__}",
    },
])
model_summary_df = pd.DataFrame([
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "catboost_tuned",
        "role": "primary_final_refit_candidate_warning_heavy",
        "phase10_oof_auc": cat_auc,
        "delta_vs_m0": cat_auc - m0_auc,
        "delta_vs_m1_baseline": cat_auc - m1_auc,
        "submission_path": str(paths["catboost_submission"].relative_to(REPO_ROOT)),
        "submission_sha256": validation_records[0]["submission_sha256"],
        "final_winner_declared": False,
        "submission_uploaded": False,
    },
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate": "m1_logistic_regression_baseline",
        "role": "fallback_reference_candidate",
        "phase10_oof_auc": m1_auc,
        "delta_vs_m0": m1_auc - m0_auc,
        "delta_vs_m1_baseline": 0.0,
        "submission_path": str(paths["m1_submission"].relative_to(REPO_ROOT)),
        "submission_sha256": validation_records[1]["submission_sha256"],
        "final_winner_declared": False,
        "submission_uploaded": False,
    },
])

candidate_log_df = pd.DataFrame([
    {
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "authorized_commit": AUTHORIZED_HEAD,
        "phase10_acceptance_commit": PHASE10_ACCEPTANCE_HEAD,
        "candidate_family": row["model_family"],
        "candidate_variant": row["candidate"],
        "final_refit_candidate": True,
        "fallback_candidate": row["candidate"] == "m1_logistic_regression_baseline",
        "features_used": "F2",
        "School_used_as_feature": False,
        "external_data_used": False,
        "leaderboard_used_for_selection": False,
        "submission_created": True,
        "submission_uploaded": False,
        "model_hyperparameters_source": row["hyperparameters_source"],
        "feature_contract": "F2 only; School excluded; official data only",
        "preprocessing_contract": "fit on full train only; test transform-only",
        "train_rows": len(train),
        "test_rows": len(test),
        "submission_rows": 696,
        "submission_path": str((paths["catboost_submission"] if row["candidate"] == "catboost_tuned" else paths["m1_submission"]).relative_to(REPO_ROOT)),
        "submission_sha256": validation_records[0 if row["candidate"] == "catboost_tuned" else 1]["submission_sha256"],
        "artifact_manifest_path": str(paths["artifact_manifest"].relative_to(REPO_ROOT)),
        "validation_report_path": str(paths["validation_report"].relative_to(REPO_ROOT)),
        "submission_validation_status": "pass",
        "manual_edit_detected": False,
        "notes": "No upload; no final winner; awaiting independent Opus review and project-director acceptance.",
    }
    for _, row in final_refit_df.iterrows()
])

candidate_selection_df.to_csv(paths["candidate_selection"], index=False)
final_refit_df.to_csv(paths["final_refit"], index=False)
submission_validation_df.to_csv(paths["submission_validation"], index=False)
model_summary_df.to_csv(paths["model_summary"], index=False)
candidate_log_df.to_csv(paths["candidate_log"], index=False)

report_lines = [
    "# Phase 11 Submission Readiness Validation Report",
    "",
    "## Executive Summary",
    "",
    "Phase 11 generated and validated two candidate submission artifacts under Option C. No upload was performed, no leaderboard feedback was used, and no final winner was declared.",
    "",
    f"- HEAD: `{head}`",
    f"- Run ID: `{RUN_ID}`",
    "- Candidate 1: `catboost_tuned` (primary final-refit candidate, warning-heavy, written waiver granted)",
    "- Candidate 2: `m1_logistic_regression_baseline` (fallback/reference candidate)",
    "- Feature set: F2 only; School excluded",
    "- Test inference: transform-only using preprocessing fitted on full train",
    "",
    "## Submission Validation",
    "",
    markdown_table(submission_validation_df, ["candidate", "submission_rows", "min_probability", "max_probability", "submission_sha256", "validation_status"]),
    "",
    "## Warnings Carried Forward",
    "",
    "- CatBoost tuned has the best global Phase 10 OOF AUC but does not clear the historical promotion bar over M1 baseline.",
    "- CatBoost tuned has only 3/5 positive folds vs M1 baseline and lacks repeated-CV stability confirmation.",
    "- Slice warnings include Age_missing=1, Position=QB, Year 2011, available_measurement_count=0, OG and OLB.",
    "- Upload order remains a manual project-director decision; last uploaded file determines final ranking.",
    "",
    "## Explicit Non-Actions",
    "",
    "- No automatic upload.",
    "- No leaderboard use.",
    "- No final winner declared.",
    "- No submission-ready model declared before Opus review and director acceptance.",
    "- No HPO, ensembles, calibration, threshold tuning, external data, or School feature.",
    "",
    "Phase 11 execution complete. Submission generated and validated, not uploaded. No final winner declared. Awaiting independent Opus submission review and project-director acceptance. Leaderboard not used for selection.",
]
paths["validation_report"].write_text("\n".join(report_lines) + "\n", encoding="utf-8")

manifest_records = []
for key in [
    "candidate_selection",
    "final_refit",
    "submission_validation",
    "model_summary",
    "validation_report",
    "candidate_log",
    "catboost_submission",
    "m1_submission",
]:
    path = paths[key]
    row_count = ""
    if path.suffix.lower() == ".csv":
        try:
            row_count = len(pd.read_csv(path))
        except Exception:
            row_count = ""
    manifest_records.append({
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "artifact_key": key,
        "path": str(path.relative_to(REPO_ROOT)),
        "sha256": sha256_file(path),
        "row_count": row_count,
        "created_utc": created_at,
    })
manifest_records.append({
    "experiment_id": EXPERIMENT_ID,
    "run_id": RUN_ID,
    "artifact_key": "artifact_manifest",
    "path": str(paths["artifact_manifest"].relative_to(REPO_ROOT)),
    "sha256": "self_excluded_until_after_write",
    "row_count": "",
    "created_utc": created_at,
})
pd.DataFrame(manifest_records).to_csv(paths["artifact_manifest"], index=False)

log_after = (REPO_ROOT / "logs" / "experiment_log.csv").read_bytes()
if log_before != log_after:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 11 execution")

status_after = subprocess.check_output(["git", "status", "--short"], cwd=REPO_ROOT, text=True)
diff_check = subprocess.run(["git", "diff", "--check"], cwd=REPO_ROOT, text=True, capture_output=True)
forbidden_diff = subprocess.check_output([
    "git", "diff", "--name-only", "--",
    "data/input",
    "notebooks/_official",
    "references",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
], cwd=REPO_ROOT, text=True)

print("Artifacts written:")
for record in manifest_records:
    print(record["path"])
print("diff_check_returncode", diff_check.returncode)
print("forbidden_diff", forbidden_diff.strip())
print("git_status_after")
print(status_after)


Artifacts written:
outputs\validation\phase11_submission_readiness_phase11_option_c_20260619_0001_candidate_selection_report.csv
outputs\validation\phase11_submission_readiness_phase11_option_c_20260619_0001_final_refit_report.csv
outputs\validation\phase11_submission_readiness_phase11_option_c_20260619_0001_submission_validation.csv
outputs\validation\phase11_submission_readiness_phase11_option_c_20260619_0001_model_summary.csv
outputs\reports\phase11_submission_readiness_phase11_option_c_20260619_0001_validation_report.md
outputs\reports\phase11_submission_readiness_phase11_option_c_20260619_0001_experiment_log_candidate.csv
outputs\submissions\phase11_submission_readiness_phase11_option_c_20260619_0001_catboost_tuned_submission.csv
outputs\submissions\phase11_submission_readiness_phase11_option_c_20260619_0001_m1_logistic_regression_baseline_submission.csv
outputs\reports\phase11_submission_readiness_phase11_option_c_20260619_0001_artifact_manifest.csv
diff_check_returncode 0
forbid

### Interpretation

- **Main result:** Phase 11 artifacts were generated and validation gates passed.
- **Methodological reading:** Candidate files are technically valid and traceable, but upload remains manual and Phase 11 acceptance requires independent Opus review.
- **Risk or warning:** CatBoost remains warning-heavy; this execution intentionally declares no winner.
- **Diagnostic decision:** Stop here and hand off to independent post-Codex Opus review.
